In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from scipy import interpolate
from scipy.signal import butter, filtfilt
from tensorflow.keras.models import load_model
from collections import Counter
from tensorflow.keras.models import load_model

/Users/didiermupenda/miniforge3/envs/dl-projects/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


# 🔮 Prediction on New Sensor Data

Load the trained model and predict activities from IMU sensor CSV files.

In [2]:
def preprocess_sensor_data_sisfall(csv_file):
    """
    Preprocess IMU sensor data for SisFall model prediction.
    
    SisFall model expects:
    - 3 features: accelerometer x, y, z only (NO gyroscope)
    - 200 timesteps per window (1 second at 200Hz)
    - Same preprocessing as SisFall paper (Sucerquia et al., 2017)
    
    Args:
        csv_file: Path to CSV with format: timestamp;seconds_elapsed;acc_z;acc_y;acc_x;gyro_z;gyro_y;gyro_x
    
    Returns:
        windows: np.array of shape (num_windows, 200, 3)
    """
    
    # Read CSV file
    df = pd.read_csv(csv_file, sep=';')
    print(f"✓ Loaded {len(df)} samples from {csv_file}")
    
    # Extract ONLY accelerometer data (x, y, z)
    # YOUR SENSOR: X=vertical(+1g gravity), Y=horizontal, Z=horizontal  
    # SISFALL:     Y=vertical(-1g gravity), X=horizontal, Z=horizontal
    # Axis remapping needed:
    #   Your acc_x (+1g) → SisFall ACCY (flip sign: -1g)
    #   Your acc_y → SisFall ACCX
    #   Your acc_z → SisFall ACCZ
    acc_data_raw = df[['acc_x', 'acc_y', 'acc_z']].values
    acc_data = np.column_stack([
        acc_data_raw[:, 1],   # your Y → sisfall X
        -acc_data_raw[:, 0],  # your X (flipped) → sisfall Y  
        acc_data_raw[:, 2]    # your Z → sisfall Z
    ])
    
    # UPSAMPLE from 100Hz to 200Hz using linear interpolation
    # print(f"  Original samples: {len(acc_data)} @ 100Hz")
    
    # Create original time indices (100Hz = 0.01s per sample)
    original_time = np.arange(len(acc_data)) * 0.01
    
    # Create upsampled time indices (200Hz = 0.005s per sample)
    upsampled_time = np.arange(0, original_time[-1], 0.005)
    
    # Interpolate each axis
    acc_upsampled = np.zeros((len(upsampled_time), 3))
    for i in range(3):
        f = interpolate.interp1d(original_time, acc_data[:, i], kind='linear')
        acc_upsampled[:, i] = f(upsampled_time)
    
    # print(f"  Upsampled samples: {len(acc_upsampled)} @ 200Hz")
    
    # Apply same preprocessing as SisFall training:
    # NOTE: Training used raw ADC * (32.0/8192.0), but our sensor outputs g-force directly
    # So we skip that conversion - data is already in the right scale
    
    # 1. LOW-PASS FILTER: 4th order Butterworth with 5 Hz cutoff
    # This is EXACTLY what SisFall paper used (Sucerquia et al., 2017, Section 3.4.1)
    def butter_lowpass_filter(data, cutoff=5, fs=200, order=4):
        """4th order IIR Butterworth low-pass filter with 5 Hz cutoff"""
        nyquist = 0.5 * fs
        normal_cutoff = cutoff / nyquist
        b, a = butter(order, normal_cutoff, btype='low', analog=False)
        return filtfilt(b, a, data, axis=0)
    
    acc_filtered = butter_lowpass_filter(acc_upsampled, cutoff=5, fs=200, order=4)
    # print(f"  Applied 4th order Butterworth low-pass filter (5Hz cutoff)")
    
    # 2. Normalize to (-1, 1) range using MinMaxScaler
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler(feature_range=(-1, 1))
    acc_scaled = scaler.fit_transform(acc_filtered)
    
    # print(f"  Applied MinMaxScaler(-1, 1)")
    # print(f"  Data range: [{acc_scaled.min():.3f}, {acc_scaled.max():.3f}]")
    
    # Create sliding windows
    TIME_STEPS = 200  # 1 second at 200Hz (matching training data)
    STEP = 40         # 0.2 seconds overlap
    
    windows = []
    for i in range(0, len(acc_scaled) - TIME_STEPS, STEP):
        window = acc_scaled[i:i + TIME_STEPS]
        windows.append(window)
    
    windows = np.array(windows, dtype=np.float32)
    
    print(f"✓ Created {len(windows)} windows")
    print(f"  Window shape: {windows.shape}")
    print(f"  Expected: (num_windows, 200, 3)")
    print(f"  Temporal duration: 1 second per window @ 200Hz")
    
    return windows

print("✓ Preprocessing function defined (FIXED: sensor data already in g-force)")

✓ Preprocessing function defined (FIXED: sensor data already in g-force)


In [3]:
def predict_sisfall_activity(csv_file, model_path='saved_model/1'):
    """
    Predict fall/activity from sensor data using trained SisFall model.
    
    Args:
        csv_file: Path to sensor CSV file
        model_path: Path to saved model (default: 'saved_model/1')
    
    Returns:
        activity: Predicted activity class
        confidence: Average confidence percentage
    """
    
    # Activity labels: Model outputs 0-indexed classes (0-9)
    # Classes 0-4: ADLs (Activities of Daily Living) - D01-D05
    # Classes 5-9: Falls - F01-F05
    ACTIVITY_LABELS = {
        0: 'D01 (ADL)',      # e.g., Walking
        1: 'D02 (ADL)',      # e.g., Jogging
        2: 'D03 (ADL)',      # e.g., Standing
        3: 'D04 (ADL)',      # e.g., Sitting
        4: 'D05 (ADL)',      # e.g., Lying
        5: 'F01 (Fall)',     # e.g., Forward fall
        6: 'F02 (Fall)',     # e.g., Backward fall
        7: 'F03 (Fall)',     # e.g., Lateral fall
        8: 'F04 (Fall)',     # e.g., Fall from sitting
        9: 'F05 (Fall)'      # e.g., Fall from standing
    }


    # ACTIVITY_LABELS = {
    #     1: 'F01',
    #     2: 'F02', 
    #     3: 'F03',
    #     4: 'F04',
    #     5: 'F05',
    #     6: 'D01',
    #     7: 'D02',
    #     8: 'D03',
    #     9: 'D04',
    #     10: 'D05'
    # }

    
    # # Preprocess sensor data
    # print("\n" + "="*50)
    # print("PREPROCESSING SENSOR DATA")
    # print("="*50)
    windows = preprocess_sensor_data_sisfall(csv_file)
    
    # # Load trained model
    # print("\n" + "="*50)
    # print("LOADING MODEL")
    # print("="*50)
    model = load_model(model_path)
    print(f"✓ Model loaded from {model_path}")
    
    # Make predictions
    # print("\n" + "="*50)
    # print("MAKING PREDICTIONS")
    # print("="*50)
    predictions = model.predict(windows, verbose=0)
    predicted_classes = np.argmax(predictions, axis=1)
    
    # Get most common prediction (voting)
    vote_counts = Counter(predicted_classes)
    most_common_class = vote_counts.most_common(1)[0][0]
    vote_count = vote_counts.most_common(1)[0][1]
    
    # Calculate confidence
    confidences = [predictions[i][predicted_classes[i]] for i in range(len(windows))]
    avg_confidence = np.mean(confidences) * 100
    
    # Get activity name
    activity_name = ACTIVITY_LABELS.get(most_common_class, f"Class_{most_common_class}")
    
    # Print results
    print(f"\n{'='*50}")
    print(f"PREDICTION RESULTS")
    print(f"{'='*50}")
    print(f"\n  Activity: {activity_name}")
    print(f"  Confidence: {avg_confidence:.1f}%")
    print(f"  Windows: {vote_count}/{len(windows)} voted for this class")
    print(f"\n  Vote breakdown:")
    for cls, count in vote_counts.most_common():
        pct = count / len(windows) * 100
        label = ACTIVITY_LABELS.get(cls, f"Class_{cls}")
        print(f"    {label:20s}: {count:3d} ({pct:.1f}%)")
    print(f"{'='*50}\n")
    
    return activity_name, avg_confidence

print("✓ Prediction function defined")

✓ Prediction function defined


In [4]:
# Load the trained SisFall model (.keras format)
# Note: Model is in parent directory's model folder
model = load_model('../model/model_v011.keras')
model.summary()

print(f"\n✓ Model loaded successfully!")
print(f"Input shape: {model.input_shape}")
print(f"Output shape: {model.output_shape}")
print(f"Number of classes: {model.output_shape[-1]}")

/Users/didiermupenda/miniforge3/envs/dl-projects/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 23 variables whereas the saved optimizer has 44 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ batch_normalization             │ (None, 200, 3)         │            12 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 200, 256)       │       266,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 200, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 200, 128)       │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 200, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 50)             │        35,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         6,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,017,048 (3.88 MB)

 Trainable params: 508,008 (1.94 MB)

 Non-trainable params: 1,030 (4.02 KB)

 Optimizer params: 508,010 (1.94 MB)


✓ Model loaded successfully!
Input shape: (None, 200, 3)
Output shape: (None, 10)
Number of classes: 10


## 🧪 Example: Predict from Your Sensor Data

Run prediction on your IMU sensor CSV file:

In [11]:
#cv_path = 'sensor_data/Calibrated/fall_backward.csv'
#cv_path = 'sensor_data/Calibrated/fall_forward.csv'

#cv_path = 'sensor_data/Recorded/falling_b_20260121_205836.csv'
cv_path = 'sensor_data/Recorded/falling_f_20260121_205736.csv'

# Example: Predict activity from your sensor data
activity, confidence = predict_sisfall_activity(
    csv_file=f'../{cv_path}',
    model_path='../model/model_v011.keras'  # or 'my_model.h5' if you saved as .h5
)

print(f"\n✓ Final prediction: {activity} ({confidence:.1f}% confidence)")

✓ Loaded 709 samples from ../sensor_data/Recorded/falling_f_20260121_205736.csv
✓ Created 31 windows
  Window shape: (31, 200, 3)
  Expected: (num_windows, 200, 3)
  Temporal duration: 1 second per window @ 200Hz


/Users/didiermupenda/miniforge3/envs/dl-projects/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 23 variables whereas the saved optimizer has 44 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


✓ Model loaded from ../model/model_v011.keras

PREDICTION RESULTS

  Activity: F05 (Fall)
  Confidence: 85.0%
  Windows: 19/31 voted for this class

  Vote breakdown:
    F05 (Fall)          :  19 (61.3%)
    D04 (ADL)           :   7 (22.6%)
    F04 (Fall)          :   5 (16.1%)


✓ Final prediction: F05 (Fall) (85.0% confidence)


## 🔍 Batch Testing & Debugging

Test multiple files to identify patterns in misclassifications:

In [46]:
# Test multiple files and compare predictions
test_files = {
    'Fall Backward': 'sensor_data/fall_backward.csv',
    'Fall Forward': 'sensor_data/fall_forward.csv',
    'Walk + Fall Forward': 'sensor_data/walk_fall-forw.csv',
    'Walk + Fall Backward': 'sensor_data/walk_fall-back.csv',
    'Walk Slowly': 'sensor_data/walk_slowly.csv',
    'Walk Quickly': 'sensor_data/walk_quickly.csv',
}

print("="*70)
print("BATCH PREDICTION RESULTS")
print("="*70)

results = []
for expected_activity, file_path in test_files.items():
    print(f"\n📁 Testing: {expected_activity}")
    print(f"   File: {file_path}")
    
    try:
        activity, confidence = predict_sisfall_activity(
            csv_file=f'../{file_path}',
            model_path='../model/model_v011.keras'
        )
        
        is_correct = "✅" if expected_activity.upper() in activity.upper() or activity in expected_activity else "❌"
        results.append({
            'Expected': expected_activity,
            'Predicted': activity,
            'Confidence': confidence,
            'Correct': is_correct
        })
        
        print(f"   {is_correct} Expected: {expected_activity} | Predicted: {activity} ({confidence:.1f}%)")
        
    except Exception as e:
        print(f"   ❌ ERROR: {str(e)}")
        results.append({
            'Expected': expected_activity,
            'Predicted': f"ERROR: {str(e)}",
            'Confidence': 0,
            'Correct': '❌'
        })

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
correct_count = sum(1 for r in results if r['Correct'] == '✅')
total_count = len(results)
accuracy = (correct_count / total_count * 100) if total_count > 0 else 0

print(f"Accuracy: {correct_count}/{total_count} ({accuracy:.1f}%)")
print("\nDetailed Results:")
for r in results:
    print(f"{r['Correct']} {r['Expected']:25s} → {r['Predicted']:20s} ({r['Confidence']:.1f}%)")

BATCH PREDICTION RESULTS

📁 Testing: Fall Backward
   File: sensor_data/fall_backward.csv
✓ Loaded 536 samples from ../sensor_data/fall_backward.csv
✓ Created 22 windows
  Window shape: (22, 200, 3)
  Expected: (num_windows, 200, 3)
  Temporal duration: 1 second per window @ 200Hz


/Users/didiermupenda/miniforge3/envs/dl-projects/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 23 variables whereas the saved optimizer has 44 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


✓ Model loaded from ../model/model_v011.keras

PREDICTION RESULTS

  Activity: D04 (ADL)
  Confidence: 79.2%
  Windows: 17/22 voted for this class

  Vote breakdown:
    D04 (ADL)           :  17 (77.3%)
    F05 (Fall)          :   4 (18.2%)
    F02 (Fall)          :   1 (4.5%)

   ❌ Expected: Fall Backward | Predicted: D04 (ADL) (79.2%)

📁 Testing: Fall Forward
   File: sensor_data/fall_forward.csv
✓ Loaded 1678 samples from ../sensor_data/fall_forward.csv
✓ Created 79 windows
  Window shape: (79, 200, 3)
  Expected: (num_windows, 200, 3)
  Temporal duration: 1 second per window @ 200Hz
✓ Model loaded from ../model/model_v011.keras


/Users/didiermupenda/miniforge3/envs/dl-projects/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 23 variables whereas the saved optimizer has 44 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



PREDICTION RESULTS

  Activity: D04 (ADL)
  Confidence: 99.1%
  Windows: 74/79 voted for this class

  Vote breakdown:
    D04 (ADL)           :  74 (93.7%)
    F01 (Fall)          :   4 (5.1%)
    F04 (Fall)          :   1 (1.3%)

   ❌ Expected: Fall Forward | Predicted: D04 (ADL) (99.1%)

📁 Testing: Walk + Fall Forward
   File: sensor_data/walk_fall-forw.csv
✓ Loaded 837 samples from ../sensor_data/walk_fall-forw.csv
✓ Created 37 windows
  Window shape: (37, 200, 3)
  Expected: (num_windows, 200, 3)
  Temporal duration: 1 second per window @ 200Hz
✓ Model loaded from ../model/model_v011.keras


...


PREDICTION RESULTS

  Activity: D04 (ADL)
  Confidence: 68.8%
  Windows: 25/50 voted for this class

  Vote breakdown:
    D04 (ADL)           :  25 (50.0%)
    F05 (Fall)          :  17 (34.0%)
    F03 (Fall)          :   5 (10.0%)
    D02 (ADL)           :   1 (2.0%)
    D03 (ADL)           :   1 (2.0%)
    F02 (Fall)          :   1 (2.0%)

   ❌ Expected: Walk + Fall Backward | Predicted: D04 (ADL) (68.8%)

📁 Testing: Walk Slowly
   File: sensor_data/walk_slowly.csv
✓ Loaded 11335 samples from ../sensor_data/walk_slowly.csv
✓ Created 562 windows
  Window shape: (562, 200, 3)
  Expected: (num_windows, 200, 3)
  Temporal duration: 1 second per window @ 200Hz
✓ Model loaded from ../model/model_v011.keras


/Users/didiermupenda/miniforge3/envs/dl-projects/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 23 variables whereas the saved optimizer has 44 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



PREDICTION RESULTS

  Activity: F05 (Fall)
  Confidence: 84.5%
  Windows: 200/562 voted for this class

  Vote breakdown:
    F05 (Fall)          : 200 (35.6%)
    F03 (Fall)          : 121 (21.5%)
    F04 (Fall)          :  98 (17.4%)
    F01 (Fall)          :  65 (11.6%)
    D04 (ADL)           :  40 (7.1%)
    F02 (Fall)          :  32 (5.7%)
    D05 (ADL)           :   3 (0.5%)
    D03 (ADL)           :   3 (0.5%)

   ❌ Expected: Walk Slowly | Predicted: F05 (Fall) (84.5%)

📁 Testing: Walk Quickly
   File: sensor_data/walk_quickly.csv
✓ Loaded 782 samples from ../sensor_data/walk_quickly.csv
✓ Created 35 windows
  Window shape: (35, 200, 3)
  Expected: (num_windows, 200, 3)
  Temporal duration: 1 second per window @ 200Hz
✓ Model loaded from ../model/model_v011.keras


/Users/didiermupenda/miniforge3/envs/dl-projects/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 23 variables whereas the saved optimizer has 44 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



PREDICTION RESULTS

  Activity: F04 (Fall)
  Confidence: 84.0%
  Windows: 8/35 voted for this class

  Vote breakdown:
    F04 (Fall)          :   8 (22.9%)
    F03 (Fall)          :   8 (22.9%)
    F05 (Fall)          :   6 (17.1%)
    F02 (Fall)          :   5 (14.3%)
    F01 (Fall)          :   4 (11.4%)
    D04 (ADL)           :   4 (11.4%)

   ❌ Expected: Walk Quickly | Predicted: F04 (Fall) (84.0%)

SUMMARY
Accuracy: 0/6 (0.0%)

Detailed Results:
❌ Fall Backward             → D04 (ADL)            (79.2%)
❌ Fall Forward              → D04 (ADL)            (99.1%)
❌ Walk + Fall Forward       → F01 (Fall)           (68.4%)
❌ Walk + Fall Backward      → D04 (ADL)            (68.8%)
❌ Walk Slowly               → F05 (Fall)           (84.5%)
❌ Walk Quickly              → F04 (Fall)           (84.0%)


### 🔧 Diagnostic: Check Data Characteristics

Analyze the raw sensor data to identify potential issues:

In [31]:
def diagnose_sensor_data(csv_file):
    """Analyze sensor data characteristics to identify issues"""
    df = pd.read_csv(csv_file, sep=';')
    acc_data = df[['acc_x', 'acc_y', 'acc_z']].values
    
    # Data is ALREADY in g-force (no conversion needed)
    acc_g = acc_data
    
    print(f"\n📊 Data Diagnostics for: {csv_file}")
    print("="*60)
    print(f"Samples: {len(df)}")
    print(f"Duration: {len(df) / 100:.2f} seconds @ 100Hz")
    
    print(f"\n📈 Accelerometer Stats (in g-force):")
    print(f"  X-axis: min={acc_g[:, 0].min():.3f}g, max={acc_g[:, 0].max():.3f}g, mean={acc_g[:, 0].mean():.3f}g")
    print(f"  Y-axis: min={acc_g[:, 1].min():.3f}g, max={acc_g[:, 1].max():.3f}g, mean={acc_g[:, 1].mean():.3f}g")
    print(f"  Z-axis: min={acc_g[:, 2].min():.3f}g, max={acc_g[:, 2].max():.3f}g, mean={acc_g[:, 2].mean():.3f}g")
    
    # Calculate magnitude
    magnitude = np.sqrt(np.sum(acc_g**2, axis=1))
    print(f"\n  Magnitude: min={magnitude.min():.3f}g, max={magnitude.max():.3f}g, mean={magnitude.mean():.3f}g")
    
    # Check for potential issues
    print(f"\n✅ Data Quality Checks:")
    
    # Check 1: Verify gravity is present (~1g)
    y_mean = abs(acc_g[:, 1].mean())
    if 0.8 <= y_mean <= 1.2:
        print(f"  ✅ Y-axis mean ({y_mean:.2f}g) ≈ 1g (gravity present)")
    elif y_mean < 0.5:
        print(f"  ❌ Y-axis mean ({y_mean:.2f}g) far from ±1g - axis orientation issue")
    else:
        print(f"  ⚠️  Y-axis mean ({y_mean:.2f}g) - check if standing still")
    
    # Check 2: Reasonable magnitude range
    if 0.8 <= magnitude.mean() <= 2.0:
        print(f"  ✅ Average magnitude {magnitude.mean():.2f}g is reasonable")
    else:
        print(f"  ⚠️  Average magnitude {magnitude.mean():.2f}g outside normal range")
    
    # Check 3: Peak values for falls
    if magnitude.max() > 3.0:
        print(f"  ✅ Peak magnitude {magnitude.max():.2f}g indicates high-impact event (fall)")
    elif magnitude.max() > 1.5:
        print(f"  ✅ Peak magnitude {magnitude.max():.2f}g indicates moderate activity")
    else:
        print(f"  ℹ️  Peak magnitude {magnitude.max():.2f}g (low activity/standing)")
    
    return acc_g

# Diagnose a few files
print("\n" + "🔍 DIAGNOSING SENSOR DATA (FIXED)" + "\n")
for name, path in [('Fall Forward', 'sensor_data/fall_forward.csv'), 
                   ('Walk Slowly', 'sensor_data/walk_slowly.csv')]:
    diagnose_sensor_data(f'../{path}')


🔍 DIAGNOSING SENSOR DATA (FIXED)


📊 Data Diagnostics for: ../sensor_data/fall_forward.csv
Samples: 1678
Duration: 16.78 seconds @ 100Hz

📈 Accelerometer Stats (in g-force):
  X-axis: min=-2.975g, max=0.408g, mean=-0.981g
  Y-axis: min=-7.607g, max=1.669g, mean=-0.036g
  Z-axis: min=-1.719g, max=2.405g, mean=0.182g

  Magnitude: min=0.234g, max=7.790g, mean=1.032g

✅ Data Quality Checks:
  ❌ Y-axis mean (0.04g) far from ±1g - axis orientation issue
  ✅ Average magnitude 1.03g is reasonable
  ✅ Peak magnitude 7.79g indicates high-impact event (fall)

📊 Data Diagnostics for: ../sensor_data/walk_slowly.csv
Samples: 11335
Duration: 113.35 seconds @ 100Hz

📈 Accelerometer Stats (in g-force):
  X-axis: min=-0.536g, max=0.703g, mean=0.124g
  Y-axis: min=-2.147g, max=-0.506g, mean=-0.983g
  Z-axis: min=-0.397g, max=1.251g, mean=0.136g

  Magnitude: min=0.549g, max=2.220g, mean=1.027g

✅ Data Quality Checks:
  ✅ Y-axis mean (0.98g) ≈ 1g (gravity present)
  ✅ Average magnitude 1.03g is reasonab

### 🔄 Testing Axis Remapping

The model might expect different axis orientation than your sensor provides.

In [35]:
# Check axis orientation for standing vs walking
print("📍 AXIS ORIENTATION CHECK\n")

# Standing: gravity should be on one axis
standing_df = pd.read_csv('../sensor_data/standing_data.csv', sep=';')
print(f"Standing (gravity aligned):")
print(f"  acc_x mean: {standing_df['acc_x'].mean():.3f}g")
print(f"  acc_y mean: {standing_df['acc_y'].mean():.3f}g")
print(f"  acc_z mean: {standing_df['acc_z'].mean():.3f}g")
print(f"  → Gravity on X-axis ({standing_df['acc_x'].mean():.2f}g)")

# Walking: should show variation
walk_df = pd.read_csv('../sensor_data/walk_slowly.csv', sep=';')
print(f"\nWalking slowly:")
print(f"  acc_x: min={walk_df['acc_x'].min():.2f}g, max={walk_df['acc_x'].max():.2f}g, std={walk_df['acc_x'].std():.2f}g")
print(f"  acc_y: min={walk_df['acc_y'].min():.2f}g, max={walk_df['acc_y'].max():.2f}g, std={walk_df['acc_y'].std():.2f}g")
print(f"  acc_z: min={walk_df['acc_z'].min():.2f}g, max={walk_df['acc_z'].max():.2f}g, std={walk_df['acc_z'].std():.2f}g")

# Fall: should show high peak
fall_df = pd.read_csv('../sensor_data/fall_forward.csv', sep=';')
mag_fall = np.sqrt(fall_df['acc_x']**2 + fall_df['acc_y']**2 + fall_df['acc_z']**2)
print(f"\nFall forward:")
print(f"  Peak magnitude: {mag_fall.max():.2f}g")
print(f"  Peak on Y-axis: {fall_df['acc_y'].min():.2f}g to {fall_df['acc_y'].max():.2f}g")

print(f"\n💡 INSIGHT:")
print(f"Your sensor: Gravity on X-axis (phone orientation)")
print(f"SisFall data: Need to check training data orientation")

📍 AXIS ORIENTATION CHECK

Standing (gravity aligned):
  acc_x mean: 1.001g
  acc_y mean: 0.020g
  acc_z mean: -0.005g
  → Gravity on X-axis (1.00g)

Walking slowly:
  acc_x: min=-0.54g, max=0.70g, std=0.16g
  acc_y: min=-2.15g, max=-0.51g, std=0.19g
  acc_z: min=-0.40g, max=1.25g, std=0.18g

Fall forward:
  Peak magnitude: 7.79g
  Peak on Y-axis: -7.61g to 1.67g

💡 INSIGHT:
Your sensor: Gravity on X-axis (phone orientation)
SisFall data: Need to check training data orientation
